In [ ]:
"""
Challenge miner for RQ1: explicit "challenge" themes with counts and examples.
- Reads your manual sheet (stratified sample) + cohort_table.csv
- (Optionally) pulls recent GitHub Actions logs for each repo
- Scans logs/YAML for challenge patterns
- Aggregates weighted counts (N_h / n_h) and saves examples

Outputs (under ...\Stratified Sample\Metrics):
  - challenge_counts_weighted.csv
  - challenge_counts_by_provider_weighted.csv
  - challenge_examples.csv
"""

import os, re, io, zipfile, json, time
import pandas as pd
import numpy as np
from pathlib import Path

# ----------------- CONFIG -----------------
BASE_DIR   = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample"
OUT_DIR    = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\Metrics"
MANUAL_FILES = [
    "Manual_review_sheet_autofilled.csv",
    "Manual_Review_Sheet__Prefilled_RQ1.csv",
]
COHORT_FILE = "cohort_table.csv"

# Log fetching (GitHub Actions)
ENABLE_GITHUB_FETCH = True
# ----------------- TOKEN LOADING -----------------


# Point to your tokens file (edit if needed)
TOKENS_ENV_PATH = Path(r"C:\Android Mobile App\All_Tokens.env")

def load_tokens_from_env_file(path: Path) -> dict:
    """
    Reads simple KEY=VALUE lines from an .env-like file.
    - Ignores blank lines and lines starting with # or ;
    - Strips surrounding quotes.
    """
    tokens = {}
    try:
        if path and path.exists():
            for raw in path.read_text(encoding="utf-8", errors="ignore").splitlines():
                line = raw.strip()
                if not line or line.startswith("#") or line.startswith(";"):
                    continue
                if "=" not in line:
                    continue
                k, v = line.split("=", 1)
                k = k.strip()
                v = v.strip().strip('"').strip("'")
                if k:
                    tokens[k] = v
    except Exception:
        pass
    return tokens

# Preferred order:
#   1) All_Tokens.env -> GITHUB_TOKEN_1
#   2) OS env var     -> GITHUB_TOKEN_1
#   3) OS env var     -> GITHUB_TOKEN (generic)
def resolve_github_token() -> str:
    file_tokens = load_tokens_from_env_file(TOKENS_ENV_PATH)
    tok = file_tokens.get("GITHUB_TOKEN_1", "") or \
          os.environ.get("GITHUB_TOKEN_1", "") or \
          os.environ.get("GITHUB_TOKEN", "")
    return tok.strip()

GITHUB_TOKEN = resolve_github_token()

MAX_RUNS_PER_REPO = 5  # cap to keep it quick
REQUEST_TIMEOUT = 30

# If you already have logs locally, point here; files named: owner.repo__provider++<anything>.log
LOCAL_LOGS_DIR = None  # e.g., r"C:\path\to\downloaded_logs"

# ----------------- TAXONOMY -----------------
THEMES = {
  "EMULATOR_BOOT_TIMEOUT": ["device offline", "boot completed timeout", "emulator: ERROR", "emulator: Panic", "Adb connection refused"],
  "HW_ACCEL/KVM_MISSING": ["/dev/kvm permission denied", "KVM is required", "accel not installed", "HAXM is not installed", "Hypervisor.framework is required"],
  "SDK_LICENSES": ["You have not accepted the license agreements", "licenses not accepted"],
  "IMAGE_NOT_FOUND": ["Package .* was not found", "failed to find target with hash string", r"system-images;android-\d+.*not found"],
  "JAVA/GRADLE/AGP_VERSION_DRIFT": ["Minimum supported Gradle is", "This version of the Android Gradle plugin requires", "Unsupported Java", "Kotlin version .* is not compatible"],
  "ANDROIDX_TEST_MISMATCH": ["Could not resolve androidx.test", "Duplicate class .* found in modules", "conflict with dependency 'androidx.test'"],
  "NETWORK_FLAKE": ["Connection reset by peer", "TLS handshake timeout", "temporary failure in name resolution", "network is unreachable"],
  "VENDOR_LAB_QUOTA": ["Quota exceeded", "Billing account not configured", "insufficient tokens for"],
  "MISSING_EMULATOR_BIN": ["emulator: not found", "No emulator installed"],
  "ADB_ISSUE": ["ADB server didn't ACK", "ADB server version mismatch", "error: device offline", "more than one device/emulator"],
  "X11/HEADLESS": ["xvfb-run: error", "Cannot open display", "DISPLAY not set"],
  "SECRETS_PERMS": ["Permission denied", "No such file or directory .*keystore", "secrets.* not set", "Missing .* environment variable"],
  "GENERIC_TIMEOUT/CANCEL": ["job timed out", "The operation was canceled", "timeout exceeded"],
  "TEST_FLAKY_SIGNAL": ["Flaky", r"retrying test", r"java\.lang\.AssertionError", r"org\.junit\.ComparisonFailure"],
}

# provider patterns to find YAML quickly if you also want to scan config
YAML_HINTS = ["reactivecircus/android-emulator-runner", "sdkmanager", "avdmanager", "emulator -avd", "gcloud firebase test android run"]

# ----------------- HELPERS -----------------
def first_existing(base, names):
    base = Path(base)
    for n in names:
        p = base / n
        if p.exists():
            return str(p)
    raise FileNotFoundError(f"Manual sheet not found in {base} (looked for {names})")

def load_manual_and_cohort():
    manual = pd.read_csv(first_existing(BASE_DIR, MANUAL_FILES))
    cohort = pd.read_csv(Path(BASE_DIR) / COHORT_FILE)
    # minimal columns
    for c in ["full_name","provider","stratum"]:
        if c not in manual.columns:
            manual[c] = np.nan
    return manual, cohort

def weights_from_cohort(cohort: pd.DataFrame, manual: pd.DataFrame, truth_mask=None):
    Nh_map = dict(zip(cohort["stratum"], cohort["N"]))
    if truth_mask is None:
        df = manual.copy()
    else:
        df = manual[truth_mask].copy()
    nh = df.groupby("stratum").size().rename("n_h").to_dict()
    weights = []
    for _, r in manual.iterrows():
        s = r.get("stratum")
        N_h = Nh_map.get(s, None)
        n_h = nh.get(s, None)
        weights.append(N_h / n_h if (N_h and n_h) else np.nan)
    return np.array(weights, dtype="float64"), Nh_map

def owner_repo(full_name: str):
    # normalize
    full_name = (full_name or "").strip().strip("/").lower()
    if "/" in full_name:
        o, r = full_name.split("/", 1)
        return o, r
    return None, None

# ----------------- GITHUB ACTIONS: fetch logs -----------------
def gh_headers():
    return {"Authorization": f"Bearer {GITHUB_TOKEN}", "Accept": "application/vnd.github+json"} if GITHUB_TOKEN else {"Accept": "application/vnd.github+json"}

def gh_get(url):
    import requests
    return requests.get(url, headers=gh_headers(), timeout=REQUEST_TIMEOUT)

def fetch_gha_logs(owner, repo, max_runs=MAX_RUNS_PER_REPO):
    """
    Returns a list of tuples: (run_id, log_text) for recent runs.
    """
    out = []
    try:
        runs = gh_get(f"https://api.github.com/repos/{owner}/{repo}/actions/runs?per_page={max_runs}")
        if runs.status_code != 200: return out
        for run in runs.json().get("workflow_runs", []):
            logzip = gh_get(run["logs_url"])
            if logzip.status_code != 200: 
                continue
            zf = zipfile.ZipFile(io.BytesIO(logzip.content))
            # Concatenate all step logs for scanning
            texts = []
            for name in zf.namelist():
                try:
                    with zf.open(name) as f:
                        texts.append(f.read().decode("utf-8", errors="ignore"))
                except Exception:
                    continue
            out.append((run["id"], "\n".join(texts)))
    except Exception:
        pass
    return out

# ----------------- LOCAL LOGS -----------------
def find_local_logs(full_name):
    if not LOCAL_LOGS_DIR: return []
    safe = full_name.replace("/", ".").lower()
    logs = list(Path(LOCAL_LOGS_DIR).glob(f"{safe}__*.log"))
    out = []
    for i, p in enumerate(logs[:MAX_RUNS_PER_REPO]):
        try:
            out.append((f"local-{i+1}", p.read_text(encoding="utf-8", errors="ignore")))
        except Exception:
            continue
    return out

# ----------------- SCANNING -----------------
PAT_MAP = {k: [re.compile(p, re.I) if not p.startswith("r'") else re.compile(eval(p), re.I) for p in v] for k, v in THEMES.items()}

def scan_text_for_themes(text: str):
    found = set()
    hits = []
    L = text or ""
    for theme, regs in PAT_MAP.items():
        for rx in regs:
            m = rx.search(L)
            if m:
                found.add(theme)
                # capture a short snippet
                i = m.start()
                snippet = L[max(0, i-80): i+160].replace("\r", "")
                hits.append((theme, snippet))
                break
    return list(found), hits

def scan_yaml_hints(text: str):
    L = (text or "").lower()
    return [h for h in YAML_HINTS if h in L]

# ----------------- MAIN -----------------
def main():
    OUT = Path(OUT_DIR); OUT.mkdir(parents=True, exist_ok=True)
    manual, cohort = load_manual_and_cohort()

    # stratified weights by stratum (for CI challenges we just weigh by presence per repo)
    w_all, Nh_map = weights_from_cohort(cohort, manual)

    rows = []
    examples = []

    for idx, row in manual.iterrows():
        full = str(row.get("full_name") or "").strip()
        provider = (row.get("provider") or "unknown").split(";")[0].strip().lower()
        stratum = row.get("stratum")
        w = w_all[idx] if not np.isnan(w_all[idx]) else np.nan

        logs = []
        # prefer local logs if provided
        logs.extend(find_local_logs(full))

        # fetch GitHub Actions logs if enabled & provider suggests actions
        if ENABLE_GITHUB_FETCH and (not logs) and provider in ("github_actions","github","actions","unknown"):
            if not GITHUB_TOKEN:
                # skip silently if no token
                pass
            else:
                o, r = owner_repo(full)
                if o and r:
                    logs.extend(fetch_gha_logs(o, r))

        # also scan YAML config from your config folder if you keep them locally
        yaml_hints = []
        yaml_text = ""
        cfg_dir = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Config_Files")
        if cfg_dir.exists():
            for p in cfg_dir.glob(full.replace("/", ".").lower() + "__*++*.yml"):
                try:
                    t = p.read_text(encoding="utf-8", errors="ignore")
                    yaml_text += "\n" + t
                except Exception:
                    pass
        if yaml_text:
            yaml_hints = scan_yaml_hints(yaml_text)

        # If we have no logs at all, we still emit a row (unlabeled)
        seen_themes = set()
        if logs:
            for run_id, txt in logs:
                fthemes, fhits = scan_text_for_themes(txt)
                for th in fthemes: seen_themes.add(th)
                # keep up to 3 examples per repo
                for th, snip in fhits[:3]:
                    examples.append({
                        "full_name": full,
                        "provider": provider,
                        "stratum": stratum,
                        "theme": th,
                        "run_id": run_id,
                        "snippet": snip.strip()[:500]
                    })

        rows.append({
            "full_name": full,
            "provider": provider,
            "stratum": stratum,
            "weight": w,
            "themes": ";".join(sorted(seen_themes)) if seen_themes else "",
            "yaml_hints": ";".join(yaml_hints) if yaml_hints else "",
            "log_runs_scanned": len(logs),
        })

    df = pd.DataFrame(rows)

    # explode themes for counting
    df_exp = df.copy()
    df_exp["themes"] = df_exp["themes"].fillna("")
    df_exp = df_exp[df_exp["themes"] != ""]
    if not df_exp.empty:
        df_exp = df_exp.assign(theme=df_exp["themes"].str.split(";")).explode("theme")
    else:
        df_exp = pd.DataFrame(columns=["full_name","provider","stratum","weight","theme"])

    # Weighted counts
    if not df_exp.empty:
        grp = df_exp.groupby("theme").agg(
            weighted_count=("weight", lambda s: float(np.nansum(s))),
            repos=("full_name","nunique"),
        ).reset_index()
        total_w = float(np.nansum(df["weight"].values))
        grp["share"] = np.where(total_w>0, grp["weighted_count"]/total_w, np.nan)
        grp = grp.sort_values("weighted_count", ascending=False)
    else:
        grp = pd.DataFrame(columns=["theme","weighted_count","repos","share"])

    # By provider
    if not df_exp.empty:
        byprov = df_exp.groupby(["provider","theme"]).agg(
            weighted_count=("weight", lambda s: float(np.nansum(s))),
            repos=("full_name","nunique"),
        ).reset_index()
        # normalize by provider totals
        totals = byprov.groupby("provider")["weighted_count"].sum().rename("total_w")
        byprov = byprov.merge(totals, on="provider", how="left")
        byprov["share_in_provider"] = np.where(byprov["total_w"]>0, byprov["weighted_count"]/byprov["total_w"], np.nan)
        byprov = byprov.drop(columns=["total_w"]).sort_values(["provider","weighted_count"], ascending=[True, False])
    else:
        byprov = pd.DataFrame(columns=["provider","theme","weighted_count","repos","share_in_provider"])

    # Save
    out_dir = Path(OUT_DIR); out_dir.mkdir(parents=True, exist_ok=True)
    grp.to_csv(out_dir/"challenge_counts_weighted.csv", index=False, encoding="utf-8-sig")
    byprov.to_csv(out_dir/"challenge_counts_by_provider_weighted.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(examples).to_csv(out_dir/"challenge_examples.csv", index=False, encoding="utf-8-sig")

    print(f"Saved -> {out_dir/'challenge_counts_weighted.csv'}")
    print(f"Saved -> {out_dir/'challenge_counts_by_provider_weighted.csv'}")
    print(f"Saved -> {out_dir/'challenge_examples.csv'}")

if __name__ == "__main__":
    main()


Saved -> C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\Metrics\challenge_counts_weighted.csv
Saved -> C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\Metrics\challenge_counts_by_provider_weighted.csv
Saved -> C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\Metrics\challenge_examples.csv
